<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/11_bilgi_damitma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bilgi Damıtma (Knowledge Distillation)

Büyük bir modelin (**öğretmen**, teacher) bilgisini, çok daha küçük ve hızlı bir modele (**öğrenci**, student) aktarma tekniği. Öğrenci yalnızca doğru etiketlerden değil, öğretmenin **olasılık dağılımından** da öğrenir.

| Örnek | Öğretmen → Öğrenci | Sonuç |
|---|---|---|
| DistilBERT (2019) | BERT-base → %40 daha küçük | %60 daha hızlı, GLUE performansının ~%97'si |
| TinyBERT, MiniLM | BERT → 4–6 katman | Mobil / uç cihazlar |
| Gemma 2 2B / 9B | Daha büyük Gemma → küçük modeller (logit damıtma ile ön eğitim) | Boyutuna göre çok güçlü küçük modeller |
| DeepSeek-R1-Distill | DeepSeek-R1'in akıl yürütme izleri → Qwen / LLaMA 1.5B–70B | Küçük modellere "düşünme" yeteneği |

Bu defterde:
1. **Karanlık bilgi (dark knowledge)** ve sıcaklık (temperature) ile yumuşatılmış olasılıklar
2. Hinton damıtma kaybını sıfırdan yazmak
3. Fashion-MNIST üzerinde öğretmen CNN → küçük öğrenci: sıfırdan eğitim vs damıtma
4. Sıcaklık ve α hiperparametrelerinin etkisi
5. Ara katman (feature) damıtma
6. LLM'lerde damıtma: token seviyesi logit damıtma ve dizi seviyesi damıtma

> Colab'da `T4 GPU` önerilir (CPU'da da çalışır, sadece daha yavaş).

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

---
## 1) Karanlık bilgi: "yanlış" sınıfların olasılıkları da bilgi taşır

Bir sneaker görüntüsü için iyi eğitilmiş bir modelin çıktısı şöyle olabilir:

| Sınıf | Tişört | Pantolon | ... | **Sandalet** | Gömlek | **Sneaker** | Çanta | **Bot** |
|---|---|---|---|---|---|---|---|---|
| Olasılık | 0.0001 | 0.00001 | ... | **0.02** | 0.0001 | **0.95** | 0.001 | **0.03** |

Sert etiket (one-hot) sadece "sneaker" der. Ama öğretmenin dağılımı çok daha fazlasını söyler: *bu görüntü sandalet ve bota biraz benziyor, pantolona hiç benzemiyor.* Hinton vd. (2015) bunu **karanlık bilgi** olarak adlandırdı. Sınıflar arası benzerlik yapısı öğrenciye çok daha zengin bir öğrenme sinyali verir.

### Sıcaklık (temperature)

Softmax çıktısı genelde çok "keskin"dir, küçük olasılıklar neredeyse sıfırdır. Logitleri bir $T > 1$ sıcaklığına bölmek dağılımı **yumuşatır** ve karanlık bilgiyi görünür kılar:

$$p_i^{(T)} = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

In [ ]:
logitler = torch.tensor([1.0, -2.0, 0.5, -1.0, 2.0, 3.5, 0.0, 9.0, 1.5, 5.5])
siniflar = ["Tişört", "Pantolon", "Kazak", "Elbise", "Mont", "Sandalet", "Gömlek", "Sneaker", "Çanta", "Bot"]

fig, eksenler = plt.subplots(1, 4, figsize=(20, 3.8), sharey=True)
for ax, T in zip(eksenler, [1, 2, 5, 10]):
    p = F.softmax(logitler / T, dim=0)
    ax.bar(siniflar, p, color=["tab:orange" if s == "Sneaker" else "tab:blue" for s in siniflar])
    ax.set_title(f"T = {T}   (entropi = {-(p * p.log()).sum():.2f})")
    ax.tick_params(axis="x", rotation=60)
eksenler[0].set_ylabel("olasılık")
plt.suptitle("Sıcaklık arttıkça dağılım yumuşar, sınıflar arası benzerlikler ortaya çıkar", y=1.04)
plt.tight_layout(); plt.show()

---
## 2) Damıtma kaybı (Hinton vd., 2015)

$$\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(p_{\text{öğretmen}}^{(T)} \,\big\|\, p_{\text{öğrenci}}^{(T)}\right) \;+\; (1 - \alpha)\cdot \mathrm{CE}\!\left(y,\, p_{\text{öğrenci}}^{(1)}\right)$$

- **İlk terim (yumuşak hedef):** Öğrenci, öğretmenin yumuşatılmış dağılımını taklit eder.
- **İkinci terim (sert hedef):** Öğrenci gerçek etiketlerden de öğrenir.
- **$T^2$ çarpanı:** Sıcaklığa bölmek gradyanları $1/T^2$ oranında küçültür; bu çarpan iki terimin ölçeğini dengeler.
- $\alpha$ genellikle 0.5–0.9, $T$ genellikle 2–10 arasıdır.

In [ ]:
def damitma_kaybi(ogrenci_logit, ogretmen_logit, etiket, T=4.0, alfa=0.7):
    yumusak_ogrenci = F.log_softmax(ogrenci_logit / T, dim=-1)
    yumusak_ogretmen = F.softmax(ogretmen_logit / T, dim=-1)
    kl = F.kl_div(yumusak_ogrenci, yumusak_ogretmen, reduction="batchmean") * (T ** 2)
    ce = F.cross_entropy(ogrenci_logit, etiket)
    return alfa * kl + (1 - alfa) * ce

# Küçük kontrol: öğrenci öğretmenle aynıysa KL terimi 0 olmalı
z = torch.randn(4, 10)
y = torch.randint(0, 10, (4,))
print(f"Aynı logitler → KL kısmı: {damitma_kaybi(z, z, y, alfa=1.0):.6f}")
print(f"Farklı logitler → KL kısmı: {damitma_kaybi(torch.randn(4, 10), z, y, alfa=1.0):.4f}")

---
## 3) Deney: Fashion-MNIST

### Veri

In [ ]:
donusum = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
egitim_veri = datasets.FashionMNIST("veri", train=True, download=True, transform=donusum)
test_veri = datasets.FashionMNIST("veri", train=False, download=True, transform=donusum)

egitim_yukleyici = DataLoader(egitim_veri, batch_size=128, shuffle=True, num_workers=2)
test_yukleyici = DataLoader(test_veri, batch_size=512, num_workers=2)

fig, eksenler = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(eksenler):
    goruntu, etiket = egitim_veri[i]
    ax.imshow(goruntu[0], cmap="gray"); ax.set_title(siniflar[etiket], fontsize=8); ax.axis("off")
plt.show()

### Öğretmen ve öğrenci modelleri

In [ ]:
class Ogretmen(nn.Module):
    """Büyük CNN: ~1.1 M parametre"""
    def __init__(self):
        super().__init__()
        self.ozellik = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.siniflandirici = nn.Sequential(nn.Flatten(), nn.Linear(128 * 7 * 7, 128), nn.ReLU(),
                                            nn.Dropout(0.5), nn.Linear(128, 10))

    def forward(self, x, ozellik_dondur=False):
        f = self.ozellik(x)
        cikti = self.siniflandirici(f)
        return (cikti, f) if ozellik_dondur else cikti


class Ogrenci(nn.Module):
    """Küçük CNN: ~25 K parametre (öğretmenden ~50× küçük)"""
    def __init__(self):
        super().__init__()
        self.ozellik = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.siniflandirici = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 32), nn.ReLU(), nn.Linear(32, 10))

    def forward(self, x, ozellik_dondur=False):
        f = self.ozellik(x)
        cikti = self.siniflandirici(f)
        return (cikti, f) if ozellik_dondur else cikti


def parametre_sayisi(m):
    return sum(p.numel() for p in m.parameters())

print(f"Öğretmen: {parametre_sayisi(Ogretmen()):,} parametre")
print(f"Öğrenci : {parametre_sayisi(Ogrenci()):,} parametre  (oran: {parametre_sayisi(Ogretmen()) / parametre_sayisi(Ogrenci()):.0f}×)")

In [ ]:
@torch.no_grad()
def dogruluk(model):
    model.eval()
    dogru = toplam = 0
    for x, y in test_yukleyici:
        x, y = x.to(cihaz), y.to(cihaz)
        dogru += (model(x).argmax(-1) == y).sum().item()
        toplam += len(y)
    return dogru / toplam

@torch.no_grad()
def cikarim_suresi(model, tekrar=20):
    model.eval()
    x = torch.randn(512, 1, 28, 28, device=cihaz)
    model(x)
    if cihaz == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(tekrar):
        model(x)
    if cihaz == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / tekrar * 1000

def egit_normal(model, epok=5, lr=1e-3):
    model.to(cihaz)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(epok):
        model.train()
        for x, y in egitim_yukleyici:
            x, y = x.to(cihaz), y.to(cihaz)
            kayip = F.cross_entropy(model(x), y)
            opt.zero_grad(); kayip.backward(); opt.step()
        print(f"  epok {e + 1}: test doğruluğu = {dogruluk(model):.4f}")
    return model

### Adım 1: Öğretmeni eğit

In [ ]:
torch.manual_seed(0)
ogretmen = egit_normal(Ogretmen(), epok=8)
ogretmen.eval()
for p in ogretmen.parameters():
    p.requires_grad = False
OGRETMEN_DOGRULUK = dogruluk(ogretmen)
print(f"\nÖğretmen test doğruluğu: {OGRETMEN_DOGRULUK:.4f}")

### Öğretmenin karanlık bilgisine bakalım

In [ ]:
x, y = next(iter(test_yukleyici))
with torch.no_grad():
    ogretmen_logit = ogretmen(x[:4].to(cihaz)).cpu()

fig, eksenler = plt.subplots(2, 4, figsize=(18, 6))
for i in range(4):
    eksenler[0, i].imshow(x[i, 0], cmap="gray"); eksenler[0, i].axis("off")
    eksenler[0, i].set_title(f"Gerçek: {siniflar[y[i]]}")
    for T, renk in [(1, "tab:blue"), (5, "tab:orange")]:
        eksenler[1, i].bar(np.arange(10) + (0.2 if T == 5 else -0.2), F.softmax(ogretmen_logit[i] / T, -1),
                           width=0.4, color=renk, label=f"T = {T}")
    eksenler[1, i].set_xticks(range(10)); eksenler[1, i].set_xticklabels(siniflar, rotation=70, fontsize=8)
    eksenler[1, i].set_yscale("log"); eksenler[1, i].set_ylim(1e-6, 1.5)
eksenler[1, 0].legend(); eksenler[1, 0].set_ylabel("olasılık (log)")
plt.suptitle("Öğretmenin olasılık dağılımı: T = 5'te benzer sınıflar belirginleşiyor", y=1.01)
plt.tight_layout(); plt.show()

### Adım 2: Öğrenciyi iki farklı şekilde eğit

- **Sıfırdan:** sadece gerçek etiketlerle (cross-entropy)
- **Damıtma ile:** aynı mimari, aynı epok, ama Hinton kaybıyla

In [ ]:
def egit_damitma(ogrenci, epok=5, T=4.0, alfa=0.7, lr=1e-3, ozellik_agirligi=0.0, kopru=None):
    ogrenci.to(cihaz)
    parametreler = list(ogrenci.parameters()) + (list(kopru.parameters()) if kopru is not None else [])
    opt = torch.optim.Adam(parametreler, lr=lr)
    for e in range(epok):
        ogrenci.train()
        for x, y in egitim_yukleyici:
            x, y = x.to(cihaz), y.to(cihaz)
            with torch.no_grad():
                ogretmen_logit, ogretmen_ozellik = ogretmen(x, ozellik_dondur=True)
            ogrenci_logit, ogrenci_ozellik = ogrenci(x, ozellik_dondur=True)
            kayip = damitma_kaybi(ogrenci_logit, ogretmen_logit, y, T=T, alfa=alfa)
            if ozellik_agirligi > 0:                       # ara katman (FitNets tarzı) damıtma
                kayip = kayip + ozellik_agirligi * F.mse_loss(kopru(ogrenci_ozellik), ogretmen_ozellik)
            opt.zero_grad(); kayip.backward(); opt.step()
        print(f"  epok {e + 1}: test doğruluğu = {dogruluk(ogrenci):.4f}")
    return ogrenci

EPOK = 5
print("Öğrenci — sıfırdan eğitim:")
torch.manual_seed(1)
ogrenci_sifirdan = egit_normal(Ogrenci(), epok=EPOK)

print("\nÖğrenci — damıtma (T = 4, α = 0.7):")
torch.manual_seed(1)
ogrenci_damitilmis = egit_damitma(Ogrenci(), epok=EPOK, T=4.0, alfa=0.7)

In [ ]:
sonuclar = pd.DataFrame([
    {"Model": "Öğretmen", "Parametre": parametre_sayisi(ogretmen), "Test doğruluğu": OGRETMEN_DOGRULUK,
     "Çıkarım (ms / 512 görüntü)": cikarim_suresi(ogretmen)},
    {"Model": "Öğrenci (sıfırdan)", "Parametre": parametre_sayisi(ogrenci_sifirdan),
     "Test doğruluğu": dogruluk(ogrenci_sifirdan), "Çıkarım (ms / 512 görüntü)": cikarim_suresi(ogrenci_sifirdan)},
    {"Model": "Öğrenci (damıtılmış)", "Parametre": parametre_sayisi(ogrenci_damitilmis),
     "Test doğruluğu": dogruluk(ogrenci_damitilmis), "Çıkarım (ms / 512 görüntü)": cikarim_suresi(ogrenci_damitilmis)},
])
sonuclar["Öğretmen performansının %'si"] = (sonuclar["Test doğruluğu"] / OGRETMEN_DOGRULUK * 100).round(1)
sonuclar.round(4)

---
## 4) Sıcaklık ve α'nın etkisi

Kısa eğitimlerle (2 epok) farklı $T$ ve $\alpha$ değerlerini tarıyoruz.

In [ ]:
tarama = []
for T in [1, 2, 4, 8]:
    for alfa in [0.3, 0.7, 0.9]:
        print(f"T = {T}, α = {alfa}")
        torch.manual_seed(1)
        m = egit_damitma(Ogrenci(), epok=2, T=T, alfa=alfa)
        tarama.append({"T": T, "α": alfa, "doğruluk": dogruluk(m)})

torch.manual_seed(1)
print("Referans: sıfırdan, 2 epok")
referans = dogruluk(egit_normal(Ogrenci(), epok=2))

tablo = pd.DataFrame(tarama).pivot(index="T", columns="α", values="doğruluk")
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(tablo.values, cmap="viridis")
ax.set_xticks(range(len(tablo.columns))); ax.set_xticklabels([f"α={a}" for a in tablo.columns])
ax.set_yticks(range(len(tablo.index))); ax.set_yticklabels([f"T={t}" for t in tablo.index])
for i in range(tablo.shape[0]):
    for j in range(tablo.shape[1]):
        ax.text(j, i, f"{tablo.values[i, j]:.4f}", ha="center", va="center", color="white")
plt.colorbar(im)
ax.set_title(f"Damıtılmış öğrenci doğruluğu (2 epok)\nsıfırdan eğitilen öğrenci: {referans:.4f}")
plt.show()

---
## 5) Ara katman (feature) damıtma — FitNets

Sadece son çıktıyı değil, öğretmenin **ara katman temsillerini** de taklit ettirebiliriz. Boyutlar farklı olduğu için öğrencinin özellik haritasını öğretmeninkine eşleyen küçük bir **köprü (regressor)** katmanı ekleriz:

$$\mathcal{L}_{\text{özellik}} = \left\lVert\, r\big(f_{\text{öğrenci}}(x)\big) - f_{\text{öğretmen}}(x) \,\right\rVert^2$$

Öğrenci özelliği: (16, 7, 7) → köprü (1×1 konvolüsyon) → (128, 7, 7) = öğretmen özelliği.

In [ ]:
torch.manual_seed(1)
kopru = nn.Conv2d(16, 128, kernel_size=1).to(cihaz)
print("Öğrenci — logit + ara katman damıtma:")
ogrenci_ozellik = egit_damitma(Ogrenci(), epok=EPOK, T=4.0, alfa=0.7, ozellik_agirligi=0.5, kopru=kopru)

pd.DataFrame([
    {"Yöntem": "Sıfırdan", "Doğruluk": dogruluk(ogrenci_sifirdan)},
    {"Yöntem": "Logit damıtma", "Doğruluk": dogruluk(ogrenci_damitilmis)},
    {"Yöntem": "Logit + ara katman damıtma", "Doğruluk": dogruluk(ogrenci_ozellik)},
    {"Yöntem": "Öğretmen (üst sınır)", "Doğruluk": OGRETMEN_DOGRULUK},
]).round(4)

---
## 6) LLM'lerde damıtma

LLM'lerde iki ana yaklaşım vardır:

| | **Token seviyesi (logit) damıtma** | **Dizi seviyesi damıtma** |
|---|---|---|
| Öğrenci neyi taklit eder? | Her pozisyonda öğretmenin **tüm sözlük üzerindeki** olasılık dağılımını | Öğretmenin **ürettiği metinleri** (sert etiket olarak) |
| Gereken erişim | Öğretmenin logitlerine (açık ağırlıklı model) | Sadece öğretmenin çıktısına (API bile yeterli) |
| Tokenizer | Öğretmen ve öğrenci **aynı sözlüğü** paylaşmalı | Farklı olabilir |
| Örnek | Gemma 2, MiniLLM, DistilGPT-2 | DeepSeek-R1-Distill, Alpaca, Orca, sentetik veri ile SFT |
| Kayıp | $\sum_t \mathrm{KL}\big(p_T(\cdot \mid x_{<t}) \,\|\, p_S(\cdot \mid x_{<t})\big)$ | Standart SFT cross-entropy'si |

**KL yönü önemlidir:** İleri KL ($\mathrm{KL}(p_T\|p_S)$) öğrenciyi öğretmenin tüm modlarını kapsamaya zorlar ("ortalama" davranış); ters KL ($\mathrm{KL}(p_S\|p_T)$, MiniLLM) öğrenciyi öğretmenin en olası modlarına odaklar. Kapasitesi düşük öğrencilerde ters KL genellikle daha iyi metin üretir.

### Token seviyesi damıtma: Qwen2.5-1.5B → Qwen2.5-0.5B

Aynı ailedeki iki model aynı tokenizer'ı paylaştığı için logit damıtma yapılabilir. Aşağıdaki hücre tam bir eğitim değil, mekanizmayı gösteren kısa bir demo (birkaç adım). Gerçek bir damıtma milyonlarca token gerektirir.

> GPU gerektirir (T4 yeterli). İki model birlikte ~4 GB indirir.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

if cihaz == "cuda":
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    llm_ogretmen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct").half().to(cihaz).eval()
    llm_ogrenci = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").to(cihaz)
    for p in llm_ogretmen.parameters():
        p.requires_grad = False

    metinler = [
        "Makine öğrenmesi, bilgisayarların açıkça programlanmadan veriden öğrenmesini sağlar.",
        "Transformer mimarisi, dikkat mekanizması sayesinde uzun bağımlılıkları modelleyebilir.",
        "Bilgi damıtma, büyük bir modelin bilgisini küçük bir modele aktarır.",
        "İstanbul, Asya ile Avrupa'yı birbirine bağlayan tarihi bir şehirdir.",
    ]
    yigin = tok(metinler, return_tensors="pt", padding=True).to(cihaz)
    maske = yigin["attention_mask"][:, 1:].bool()

    def token_kl(T=2.0):
        with torch.no_grad():
            t_logit = llm_ogretmen(**yigin).logits[:, :-1].float()
        s_logit = llm_ogrenci(**yigin).logits[:, :-1].float()
        V = min(t_logit.size(-1), s_logit.size(-1))            # sözlük boyutları ufak farklılık gösterebilir
        kl = F.kl_div(F.log_softmax(s_logit[..., :V] / T, -1), F.softmax(t_logit[..., :V] / T, -1),
                      reduction="none").sum(-1)
        return (kl[maske].mean()) * T ** 2

    opt = torch.optim.AdamW(llm_ogrenci.parameters(), lr=1e-5)
    for adim in range(15):
        llm_ogrenci.train()
        kayip = token_kl()
        opt.zero_grad(); kayip.backward(); opt.step()
        if adim % 3 == 0:
            print(f"adım {adim:2d}: öğretmen–öğrenci token KL = {kayip.item():.4f}")
else:
    print("Bu demo GPU gerektiriyor. Colab'da: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU")

### Dizi seviyesi damıtma: öğretmenin cevaplarıyla SFT

Logitlere erişim yoksa (kapalı model API'si) veya tokenizer'lar farklıysa: öğretmene bir istem kümesi verilir, ürettiği cevaplar toplanır ve öğrenci bu (istem, cevap) çiftleri üzerinde standart SFT ile eğitilir. Bu sentetik veri yaklaşımıdır; DeepSeek-R1-Distill modelleri ~800 bin öğretmen örneği üzerinde bu şekilde eğitilmiştir.

```python
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

# 1) Öğretmenden cevap topla (büyük açık model veya kullanım koşullarının izin verdiği bir API)
istemler = ["Fotosentezi açıkla.", "Bir sayının asal olup olmadığını kontrol eden Python fonksiyonu yaz.", ...]
veri = [{"messages": [{"role": "user", "content": p},
                      {"role": "assistant", "content": ogretmen_cevapla(p)}]} for p in istemler]

# 2) Öğrenciyi bu veride standart SFT ile eğit
trainer = SFTTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", train_dataset=Dataset.from_list(veri),
                     args=SFTConfig(output_dir="ogrenci-sft", num_train_epochs=2, learning_rate=2e-5))
trainer.train()
```

> ⚠️ Ticari API'lerin çoğunun kullanım koşulları, çıktılarının **rakip model eğitmek** için kullanılmasını yasaklar. Damıtma verisi üretirken öğretmen modelin lisansını mutlaka kontrol edin.

---
## Özet

| Kavram | Özet |
|---|---|
| **Öğretmen–öğrenci** | Büyük modelin bilgisini küçük, hızlı modele aktarma |
| **Karanlık bilgi** | Yanlış sınıflara verilen küçük olasılıklar, sınıflar arası benzerlik yapısını taşır |
| **Sıcaklık $T$** | Dağılımı yumuşatarak karanlık bilgiyi görünür kılar; kayıpta $T^2$ ile dengelenir |
| **Hinton kaybı** | $\alpha T^2\,\mathrm{KL}(p_T^{(T)}\|p_S^{(T)}) + (1-\alpha)\,\mathrm{CE}(y, p_S)$ |
| **Ara katman damıtma** | Öğrencinin iç temsillerini köprü katmanıyla öğretmeninkilere yaklaştırma |
| **LLM: token seviyesi** | Her pozisyonda sözlük dağılımı üzerinde KL; aynı tokenizer gerekir |
| **LLM: dizi seviyesi** | Öğretmenin ürettiği metinlerle SFT; sadece çıktıya erişim yeterli |

**Damıtma, kuantizasyon ve budama (pruning)** birbirini tamamlar: önce damıt (daha küçük mimari), sonra kuantize et (daha az bit) → `08_kuantizasyon.ipynb`.

**Kaynaklar:** Hinton, Vinyals & Dean (2015) *Distilling the Knowledge in a Neural Network* · Romero vd. (2015) *FitNets* · Sanh vd. (2019) *DistilBERT* · Gu vd. (2024) *MiniLLM* · Gemma Team (2024) *Gemma 2* · DeepSeek-AI (2025) *DeepSeek-R1*